# PretrainedTEA (annual-experts variant) — control for the averaging scheme

This is PretrainedTEA built with the **TEA averaging scheme** (one expert per year) instead of
the cumulative one.

The submitted PretrainedTEA averages *cumulative* year-experts (each expert trained on all data
up to its year, with recency-weighted examples). This variant instead averages *annual* experts,
exactly as the TEA strategy does (each expert trained on a single year, no internal reweighting),
on the continued-pretrained (MLM) backbone.

Purpose: it makes the naming literal (PretrainedTEA = TEA + pretraining) and, if it holds up,
lets TEA itself serve as the averaging-only cell of the 2x2 factorial, removing the need for a
separate averaging-only ablation. Compare its four numbers (macro-F1, AUROC, seed disagreement,
Q5) against the submitted PretrainedTEA to decide which averaging scheme the proposed method
should use. One expert per year (2014, 2015, 2016), same init per seed, averaged with recency
weights (2014:0.5, 2015:0.75, 2016:1.0). Seeds 42, 1, 2.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [ ]:
import copy
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))

# TEA scheme (annual experts, one per year) on the continued-pretrained MLM backbone.
MODEL_NAME = 'tamarasuarezrod/longeval-s2-mlm-backbone'
SEEDS      = [42, 1, 2]
STRATEGY   = 'pretrainedtea-annual'
RECENCY_W  = {'2014': 0.5, '2015': 0.75, '2016': 1.0}
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## Data

In [ ]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

with open(DATA_DIR / 'train_eval/train.json') as f:
    train_full = pd.DataFrame(json.load(f)).rename(columns={'distant_label': 'label'})
train_full['year'] = train_full['created_at'].str.extract(r'(20\d\d)')
print(train_full['year'].value_counts().sort_index())

eval_df = load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label')
test_dfs = {
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}

label2id = {l: i for i, l in enumerate(sorted(train_full['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)


## Training loop (seeds 42, 1, 2)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

eval_ds  = make_dataset(eval_df)
test_ds  = {n: make_dataset(df) for n, df in test_dfs.items()}

all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*50}  SEED {SEED}')

    def train_expert(year):
        # TEA scheme: one expert per single year (NOT cumulative), no internal reweighting.
        sub = train_full[train_full['year'] == year]
        ds_yr = make_dataset(sub)
        set_seed(SEED)   # identical init for every expert (required for weight averaging)
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True)  # MLM backbone has no classifier head
        args = TrainingArguments(
            output_dir=f'/tmp/{STRATEGY}_seed{SEED}/{year}', num_train_epochs=25,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
            logging_steps=50, fp16=(DEVICE=='cuda'), seed=SEED, report_to='none')
        tr = Trainer(model=m, args=args, train_dataset=ds_yr, eval_dataset=eval_ds,
                     processing_class=tokenizer,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
        tr.train()
        return m

    experts = {yr: train_expert(yr) for yr in ['2014', '2015', '2016']}

    # Recency-weighted average of the annual experts
    total = sum(RECENCY_W.values())
    avg_sd = copy.deepcopy(experts['2016'].state_dict())
    for k in avg_sd:
        avg_sd[k] = sum(RECENCY_W[yr] * experts[yr].state_dict()[k].float()
                        for yr in RECENCY_W) / total
    avg_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)
    avg_model.load_state_dict(avg_sd)
    avg_model.to(DEVICE)

    eval_trainer = Trainer(model=avg_model,
                           args=TrainingArguments(output_dir=f'/tmp/{STRATEGY}_eval',
                                                   per_device_eval_batch_size=64,
                                                   fp16=(DEVICE=='cuda'), report_to='none'),
                           processing_class=tokenizer)

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = eval_trainer.predict(ds)
        f1 = f1_score(np.array(ds['label']), np.argmax(preds_out.predictions, axis=1), average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = avg_model


## Summary

In [ ]:
splits = ['test_within', 'test_short', 'test_long']
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short'] - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']  - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")

print()
for sp in splits:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}-{max(vals):.4f}]")


## Save to HuggingFace

In [ ]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)
